In [ ]:
import numpy as np
import torch
from torchvision import transforms
import scipy.ndimage as ndimage
from PIL import Image

In [ ]:
# Constants
MAX_HW = 384
IM_NORM_MEAN = [0.485, 0.456, 0.406]
IM_NORM_STD = [0.229, 0.224, 0.225]

TTensor = transforms.Compose([
    transforms.ToTensor(),
])

In [ ]:
class ResizeValImage:
    def __init__(self, args=None, MAX_HW=384):
        self.max_hw = MAX_HW

    def __call__(self, sample):
        image = sample['image']
        dots = sample['dots']
        m_flag = sample.get('m_flag', 0)
        lines_boxes = sample['lines_boxes']
        neg_lines_boxes = sample['neg_lines_boxes']

        W, H = image.size

        new_H = new_W = self.max_hw
        scale_factor_h = float(new_H) / H
        scale_factor_w = float(new_W) / W
        resized_image = transforms.Resize((new_H, new_W))(image)
        resized_image = TTensor(resized_image)

        # Resize density map
        resized_density = np.zeros((new_H, new_W), dtype='float32')
        for i in range(dots.shape[0]):
            resized_density[min(new_H - 1, int(dots[i][1] * scale_factor_h))] \
                           [min(new_W - 1, int(dots[i][0] * scale_factor_w))] = 1
        resized_density = ndimage.gaussian_filter(resized_density, sigma=4, order=0)
        resized_density = torch.from_numpy(resized_density) * 60

        # Crop bboxes and resize as 64x64
        boxes = list()
        rects = list()
        cnt = 0
        for box in lines_boxes:
            cnt += 1
            if cnt > 3:
                break
            box2 = [int(k) for k in box]
            y1 = int(box2[0] * scale_factor_h)
            x1 = int(box2[1] * scale_factor_w)
            y2 = int(box2[2] * scale_factor_h)
            x2 = int(box2[3] * scale_factor_w)
            rects.append(torch.tensor([y1, x1, y2, x2]))
            bbox = resized_image[:, y1:y2 + 1, x1:x2 + 1]
            bbox = transforms.Resize((64, 64))(bbox)
            boxes.append(bbox)
        boxes = torch.stack(boxes)
        pos = torch.stack(rects)
        
        neg_boxes = list()
        neg_rects = list()
        cnt = 0
        for box in neg_lines_boxes:
            cnt += 1
            if cnt > 3:
                break
            box2 = [int(k) for k in box]
            y1 = int(box2[0] * scale_factor_h)
            x1 = int(box2[1] * scale_factor_w)
            y2 = int(box2[2] * scale_factor_h)
            x2 = int(box2[3] * scale_factor_w)
            neg_rects.append(torch.tensor([y1, x1, y2, x2]))
            neg_bbox = resized_image[:, y1:y2 + 1, x1:x2 + 1]
            neg_bbox = transforms.Resize((64, 64))(neg_bbox)
            neg_boxes.append(neg_bbox)
        neg_boxes = torch.stack(neg_boxes)
        
        sample = {
            'image': resized_image,
            'boxes': boxes,
            'neg_boxes': neg_boxes,
            'pos': pos,
            'gt_density': resized_density,
            'm_flag': m_flag
        }
        return sample


def transform_val(args=None):
    return transforms.Compose([ResizeValImage(args, MAX_HW)])

In [ ]:
# Test
print("Transform function ready!")
print(f"MAX_HW: {MAX_HW}")